# Web RAG with a persistent LlamaIndex vector index
Electoral bond data: load web page, split semantically, persist index, reload and query.

In [1]:
import os

from get_api_key import get_api_key
from llama_index.core import (
    Settings,
    StorageContext,
    VectorStoreIndex,
    load_index_from_storage,
)
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.readers.web import SimpleWebPageReader

os.environ["OPENAI_API_KEY"] = get_api_key()

In [2]:
# Configure LlamaIndex settings
Settings.llm = OpenAI(model="gpt-3.5-turbo", max_tokens=300)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", dimensions=512)

In [3]:
web_urls = ["https://myneta.info/electoral_bonds/party_alphabetical.php"]
web_reader = SimpleWebPageReader(html_to_text=True)
splitter = SemanticSplitterNodeParser(
    buffer_size=1,
    breakpoint_percentile_threshold=95,
    embed_model=Settings.embed_model,
)
index_id = "web_qa"
persist_dir = "./storage"

In [4]:
def load_web_data(web_urls=web_urls, web_reader=web_reader, splitter=splitter):
    documents = web_reader.load_data(web_urls)
    return splitter.get_nodes_from_documents(documents)


web_nodes = load_web_data()
print("nodes:", len(web_nodes))

nodes: 4


In [5]:
def index_and_store_to_disk(web_nodes=web_nodes):
    index = VectorStoreIndex(web_nodes)
    index.set_index_id(index_id)
    index.storage_context.persist(persist_dir=persist_dir)
    return index


def load_index_from_disk(index_id=index_id):
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    return load_index_from_storage(storage_context, index_id=index_id)

In [6]:
index_and_store_to_disk(web_nodes)
loaded_index = load_index_from_disk()
query_engine = loaded_index.as_query_engine(response_mode="compact")
print("persisted files:", os.listdir(persist_dir))

persisted files: ['default__vector_store.json', 'docstore.json', 'graph_store.json', 'image__vector_store.json', 'index_store.json']


In [7]:
def query_web_data(qterm):
    if not qterm or not qterm.strip():
        return None
    return query_engine.query(qterm)

In [8]:
queries = [
    "List the top 3 largest Donors and their donations? Respond in JSON format.",
    "List the top 3 parties based on total donations? Respond in JSON format.",
    "",
]

for q in queries:
    print("query:", repr(q))
    print(query_web_data(q))
    print()

query: 'List the top 3 largest Donors and their donations? Respond in JSON format.'
{
    "Top 3 Donors": [
        {
            "Donor": "AVEES TRADING FINANCE PVT LTD",
            "Amount": "Rs. 10,00,00,000.00"
        },
        {
            "Donor": "BAJAJ AUTO LTD",
            "Amount": "Rs. 8,00,00,000.00"
        },
        {
            "Donor": "MKJ ENTERPRISES LIMITED",
            "Amount": "Rs. 7,00,00,000.00"
        }
    ]
}

query: 'List the top 3 parties based on total donations? Respond in JSON format.'
{
    "Top 3 Parties based on Total Donations": [
        {
            "Party": "AAM AADMI PARTY",
            "Total Donation Amount": "Rs. 50,00,00,000.00"
        },
        {
            "Party": "AAM AADMI PARTY",
            "Total Donation Amount": "Rs. 30,00,00,000.00"
        },
        {
            "Party": "AAM AADMI PARTY",
            "Total Donation Amount": "Rs. 20,00,00,000.00"
        }
    ]
}

query: ''
None

